# 05. Регресійні моделі

На цьому етапі розпочинається побудова моделей машинного навчання
для прогнозування цільової змінної `calories_burned`.

Спочатку створюється базова лінія (baseline) за допомогою класичних
регресійних моделей.

Основні завдання:

- завантаження підготовлених даних;
- побудова базових регресійних моделей;
- навчання моделей на тренувальній вибірці;
- оцінювання на валідаційній вибірці;
- порівняння отриманих результатів;
- збереження моделей та метрик.

In [1]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

import os
import sys
import importlib
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

warnings.filterwarnings("ignore")

In [2]:
# ============================================================
# CONNECT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

Mounted at /content/drive


In [3]:
# ============================================================
# LOAD PROJECT
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/FitnessML_Master"

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

importlib.invalidate_caches()

print("✓ Project connected")
print(PROJECT_DIR)

✓ Project connected
/content/drive/MyDrive/FitnessML_Master


In [4]:
# ============================================================
# IMPORT PROJECT MODULES
# ============================================================

import config
import utils

config = importlib.reload(config)
utils = importlib.reload(utils)

utils.section("Project modules")

print("✓ config.py loaded")
print("✓ utils.py loaded")


PROJECT MODULES
✓ config.py loaded
✓ utils.py loaded


In [5]:
# ============================================================
# LOAD PREPROCESSED DATA
# ============================================================

utils.section("Preprocessed dataset")

X_train = pd.read_csv(
    config.TABLES_DIR / "03_X_train.csv"
)

X_valid = pd.read_csv(
    config.TABLES_DIR / "03_X_valid.csv"
)

X_test = pd.read_csv(
    config.TABLES_DIR / "03_X_test.csv"
)

y_train = pd.read_csv(
    config.TABLES_DIR / "03_y_train.csv"
)["calories_burned"]

y_valid = pd.read_csv(
    config.TABLES_DIR / "03_y_valid.csv"
)["calories_burned"]

y_test = pd.read_csv(
    config.TABLES_DIR / "03_y_test.csv"
)["calories_burned"]

print(f"Train      : {X_train.shape}")
print(f"Validation : {X_valid.shape}")
print(f"Test       : {X_test.shape}")


PREPROCESSED DATASET
Train      : (292000, 10)
Validation : (36500, 10)
Test       : (36500, 10)


In [6]:
# ============================================================
# DATA VALIDATION
# ============================================================

utils.section("Data validation")

assert X_train.shape[0] == y_train.shape[0]
assert X_valid.shape[0] == y_valid.shape[0]
assert X_test.shape[0] == y_test.shape[0]

assert list(X_train.columns) == list(X_valid.columns)
assert list(X_train.columns) == list(X_test.columns)

assert "calories_burned" not in X_train.columns

print("✓ Feature matrices are consistent")
print("✓ Target variable is separated")
print("✓ Train / validation / test dimensions are valid")


DATA VALIDATION
✓ Feature matrices are consistent
✓ Target variable is separated
✓ Train / validation / test dimensions are valid


# Базова лінійна регресія

Першою моделлю для порівняння використовується лінійна регресія.

Вона виступає базовою моделлю (baseline), з якою надалі порівнюватимуться
складніші алгоритми.

Модель навчається на тренувальній вибірці, після чого її якість оцінюється
на валідаційній вибірці.

In [7]:
# ============================================================
# LINEAR REGRESSION
# ============================================================

utils.section("Linear Regression")

linear_model = LinearRegression()

linear_model.fit(
    X_train,
    y_train
)

y_pred_linear = linear_model.predict(
    X_valid
)

print("✓ Linear Regression trained")


LINEAR REGRESSION
✓ Linear Regression trained


# Оцінювання моделі

Для оцінювання якості регресійної моделі використовуються три метрики:

- **MAE (Mean Absolute Error)** — середня абсолютна помилка прогнозу;
- **RMSE (Root Mean Squared Error)** — корінь із середньоквадратичної помилки;
- **R² (коефіцієнт детермінації)** — частка варіації цільової змінної, пояснена моделлю.

Менше значення MAE та RMSE означає меншу помилку прогнозування.
Для R² більше значення означає кращу відповідність моделі даним.

In [8]:
# ============================================================
# MODEL EVALUATION
# ============================================================

def evaluate_regression(
    y_true,
    y_pred,
    model_name
):
    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    return {
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }


linear_results = evaluate_regression(
    y_valid,
    y_pred_linear,
    "Linear Regression"
)

display(
    pd.DataFrame([linear_results])
)

,Model,MAE,RMSE,R2
0,Linear Regression,239.690599,299.904654,-0.000097


# Дерево рішень

Дерево рішень дозволяє моделювати нелінійні залежності між ознаками
та цільовою змінною.

На відміну від лінійної регресії, модель не передбачає лінійного зв'язку
між вхідними ознаками та результатом.

In [9]:
# ============================================================
# DECISION TREE
# ============================================================

utils.section("Decision Tree")

tree_model = DecisionTreeRegressor(
    random_state=config.RANDOM_STATE
)

tree_model.fit(
    X_train,
    y_train
)

y_pred_tree = tree_model.predict(
    X_valid
)

tree_results = evaluate_regression(
    y_valid,
    y_pred_tree,
    "Decision Tree"
)

display(
    pd.DataFrame([tree_results])
)


DECISION TREE


,Model,MAE,RMSE,R2
0,Decision Tree,349.34562,437.865736,-1.131855


# Random Forest

Для подальшого порівняння використовується ансамблевий метод
Random Forest Regressor.

Модель об'єднує результати великої кількості дерев рішень,
що дозволяє зменшити залежність від окремого дерева та покращити
стабільність прогнозування.

На першому етапі використовується базова конфігурація моделі
без складного налаштування гіперпараметрів.

In [11]:
# ============================================================
# RANDOM FOREST
# ============================================================

utils.section("Random Forest")

random_forest_model = RandomForestRegressor(
    n_estimators=100,
    random_state=config.RANDOM_STATE,
    n_jobs=-1
)

random_forest_model.fit(
    X_train,
    y_train
)

y_pred_rf = random_forest_model.predict(
    X_valid
)

random_forest_results = evaluate_regression(
    y_valid,
    y_pred_rf,
    "Random Forest"
)

display(
    pd.DataFrame([random_forest_results])
)


RANDOM FOREST


,Model,MAE,RMSE,R2
0,Random Forest,242.008414,302.821678,-0.019646


In [12]:
# ============================================================
# BASELINE MODEL COMPARISON
# ============================================================

utils.section("Baseline model comparison")

baseline_results = pd.DataFrame([
    linear_results,
    tree_results,
    random_forest_results
])

display(
    baseline_results.sort_values(
        by="RMSE"
    ).reset_index(drop=True)
)


BASELINE MODEL COMPARISON


,Model,MAE,RMSE,R2
0,Linear Regression,239.690599,299.904654,-0.000097
1,Random Forest,242.008414,302.821678,-0.019646
2,Decision Tree,349.345620,437.865736,-1.131855


# Моделювання з використанням похідних ознак

Після побудови базових моделей виконується повторне моделювання
з використанням набору ознак, сформованого на етапі Feature Engineering.

Мета експерименту — визначити, чи покращують створені похідні ознаки
якість прогнозування порівняно з базовим набором ознак.

In [13]:
# ============================================================
# LOAD ENGINEERED FEATURES
# ============================================================

utils.section("Engineered dataset")

X_train_engineered = pd.read_csv(
    config.TABLES_DIR / "04_X_train_engineered.csv"
)

X_valid_engineered = pd.read_csv(
    config.TABLES_DIR / "04_X_valid_engineered.csv"
)

X_test_engineered = pd.read_csv(
    config.TABLES_DIR / "04_X_test_engineered.csv"
)

print(f"Train      : {X_train_engineered.shape}")
print(f"Validation : {X_valid_engineered.shape}")
print(f"Test       : {X_test_engineered.shape}")


ENGINEERED DATASET
Train      : (292000, 18)
Validation : (36500, 18)
Test       : (36500, 18)


In [14]:
# ============================================================
# ENGINEERED DATA VALIDATION
# ============================================================

utils.section("Engineered data validation")

assert X_train_engineered.shape[0] == y_train.shape[0]
assert X_valid_engineered.shape[0] == y_valid.shape[0]
assert X_test_engineered.shape[0] == y_test.shape[0]

assert list(X_train_engineered.columns) == list(
    X_valid_engineered.columns
)

assert "calories_burned" not in X_train_engineered.columns

print("✓ Engineered feature matrices are consistent")
print("✓ Target variable is separated")
print("✓ 18 features available")


ENGINEERED DATA VALIDATION
✓ Engineered feature matrices are consistent
✓ Target variable is separated
✓ 18 features available


In [15]:
# ============================================================
# LINEAR REGRESSION - ENGINEERED FEATURES
# ============================================================

utils.section("Linear Regression - engineered features")

linear_engineered_model = LinearRegression()

linear_engineered_model.fit(
    X_train_engineered,
    y_train
)

y_pred_linear_engineered = (
    linear_engineered_model.predict(
        X_valid_engineered
    )
)

linear_engineered_results = evaluate_regression(
    y_valid,
    y_pred_linear_engineered,
    "Linear Regression + Engineered"
)

display(
    pd.DataFrame([linear_engineered_results])
)


LINEAR REGRESSION - ENGINEERED FEATURES


,Model,MAE,RMSE,R2
0,Linear Regression + Engineered,239.702469,299.92232,-0.000215


In [16]:
# ============================================================
# LINEAR REGRESSION COMPARISON
# ============================================================

utils.section("Linear Regression comparison")

linear_comparison = pd.DataFrame([
    linear_results,
    linear_engineered_results
])

display(
    linear_comparison
)


LINEAR REGRESSION COMPARISON


,Model,MAE,RMSE,R2
0,Linear Regression,239.690599,299.904654,-0.000097
1,Linear Regression + Engineered,239.702469,299.922320,-0.000215


In [18]:
# ============================================================
# SAVE CURRENT MODEL RESULTS
# ============================================================

utils.section("Save model results")

model_results = pd.concat(
    [
        baseline_results,
        pd.DataFrame([linear_engineered_results])
    ],
    ignore_index=True
)

MODEL_RESULTS_FILE = (
    config.TABLES_DIR / "05_model_results.csv"
)

model_results.to_csv(
    MODEL_RESULTS_FILE,
    index=False
)

print(f"✓ Model results saved -> {MODEL_RESULTS_FILE}")

display(model_results)


SAVE MODEL RESULTS
✓ Model results saved -> /content/drive/MyDrive/FitnessML_Master/tables/05_model_results.csv


,Model,MAE,RMSE,R2
0,Linear Regression,239.690599,299.904654,-0.000097
1,Decision Tree,349.345620,437.865736,-1.131855
2,Random Forest,242.008414,302.821678,-0.019646
3,Linear Regression + Engineered,239.702469,299.922320,-0.000215


In [ ]:
# ============================================================
# RANDOM FOREST - ENGINEERED FEATURES
# ============================================================

utils.section("Random Forest - engineered features")

random_forest_engineered_model = RandomForestRegressor(
    n_estimators=100,
    random_state=config.RANDOM_STATE,
    n_jobs=-1
)

random_forest_engineered_model.fit(
    X_train_engineered,
    y_train
)

y_pred_rf_engineered = (
    random_forest_engineered_model.predict(
        X_valid_engineered
    )
)

random_forest_engineered_results = evaluate_regression(
    y_valid,
    y_pred_rf_engineered,
    "Random Forest + Engineered"
)

display(
    pd.DataFrame([random_forest_engineered_results])
)

In [19]:
# ============================================================
# NOTEBOOK COMPLETED
# ============================================================

utils.section("Regression modeling completed")


REGRESSION MODELING COMPLETED
